# Branch-Level Demand Forecasting Engine

Forecasts demand per **(branch, P/N)** instead of collapsing to the national "ALL" rollup, using 8 methods (MA, WMA, EWMA, Linear Regression, Polynomial Regression degree 2/3, SES, DES) and picks the best method per part by backtest MAE.

**Multi-agency:** the engine auto-detects an `Agc` column if one is present and carries it straight through to the output, so a single consolidated export covering several agencies (e.g. a `pmovdcE` file with an "All" sheet spanning agc 06, 07, 08, 10, ...) is forecast in one pass -- branch, agency, and P/N together form the row key, so the same P/N in the same branch under two different agencies is kept as two separate rows.

**Output naming:** the output columns are named to match `CNH_FD_Processed.ipynb` / `FD_CNH.ipynb` so this notebook's output can be fed straight into that pipeline without any renaming: the branch column is `brc` (not `branch`), and the national rollup rows (see Section 3) are labeled `brc='National'` (not `'ALL'`), with `agc='ALL AGC'` for the grand total across every agency.

**Input:** you set `INPUT_FILE`, `SHEET_NAME`, and `SKIP_ROWS` to match your export (see Section 1). Leave `SHEET_NAME`/`SKIP_ROWS` as `None` and they're auto-detected instead, so small format changes won't break it.

**Model selection (new):** before picking a "best" model per part, the engine now classifies each (branch, P/N) row's demand pattern (Smooth / Erratic / Intermittent / Lumpy / Dead, via the Syntetos-Boylan ADI/CV² framework), detects structural level shifts and one-off bulk-order months, and restricts the candidate model set accordingly -- so a volatile or shifting part can no longer get assigned a Poly-2/Poly-3 fit it isn't suited for just because it happened to win on backtest MAE. A forecast floor/fallback safety net also catches any remaining near-zero forecast for a part that's still actively moving. See Section 3.

**Performance:** every model is vectorized with numpy across all rows at once (no per-row Python loops, no per-row sklearn/statsmodels calls), which is what makes branch-level (~30k+ rows) practical to run in seconds instead of minutes.


In [1]:
import glob
import os
import re
import time

import numpy as np
import pandas as pd

## 1. File loading

Give it the sheet name and the number of rows to skip before the header row -- the same two things you'd set in a plain `pd.read_excel(file, sheet_name=..., skiprows=...)` call. Set either one to `None` (the default) to have it auto-detected instead, which is handy if you don't know the layout yet or it shifts slightly between exports.


In [2]:
def find_data_sheet(path):
    """Picks the sheet most likely to hold the movement data.
    Prefers a sheet whose name contains 'pmovdc'; falls back to the first sheet.
    Only used when sheet_name isn't given explicitly to load_movement_data().
    """
    xl = pd.ExcelFile(path)
    for name in xl.sheet_names:
        if 'pmovdc' in name.lower():
            return name
    return xl.sheet_names[0]


def find_header_row(path, sheet_name, max_scan=15):
    """Scans the first `max_scan` rows to find the one that looks like the
    real header row (contains a P/N column and a branch column), regardless
    of how many metadata rows precede it. Only used when skiprows isn't
    given explicitly to load_movement_data().
    """
    preview = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=max_scan)
    for i, row in preview.iterrows():
        vals = [str(v).strip().lower() for v in row.values]
        has_pn = any(v in ('p/n', 'pn', 'part number') for v in vals)
        has_branch = any(v in ('brc', 'branch', 'br') for v in vals)
        if has_pn and has_branch:
            return i
    raise ValueError(
        f"Could not auto-detect the header row in the first {max_scan} rows of "
        f"sheet '{sheet_name}'. Open the file and check where the real column "
        f"headers (P/N, Brc, etc.) start, then pass skiprows explicitly."
    )


def load_movement_data(file_path, sheet_name=None, skiprows=None):
    """Loads the file you point it at.

    sheet_name / skiprows: pass these explicitly once you know them (e.g. a
    multi-agency export like "pmovdcE_CNH_2_Jul_26.xlsx" -> sheet_name="All",
    skiprows=4) for a fast, deterministic load. Leave either as None and it's
    auto-detected instead, so the notebook still works if the export layout
    shifts slightly (extra sheets, an inserted metadata row, etc.).
    """
    sheet = sheet_name if sheet_name is not None else find_data_sheet(file_path)
    rows_to_skip = skiprows if skiprows is not None else find_header_row(file_path, sheet)
    df = pd.read_excel(file_path, sheet_name=sheet, skiprows=rows_to_skip)
    df.columns = [str(c).strip().replace(chr(10), ' ').lower() for c in df.columns]
    print(f"Loaded: {file_path}")
    print(f"  sheet: '{sheet}', skiprows: {rows_to_skip}, rows: {len(df)}")
    return df


## 2. Vectorized forecasting models

Each function takes a `(n_rows, 12)` actual-demand array and returns a `(n_rows, 13)` forecast series (12 in-sample fitted points + 1 forward forecast), computed for **all rows at once**.

In [3]:
def batch_ma(clipped_15):
    """Simple moving average, 3-point window, over a 15-length input -> 13 outputs."""
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(clipped_15, window_shape=3, axis=1)  # (n,13,3)
    return windows.mean(axis=2)


def batch_wma(clipped_15, d_last, step=0.05):
    """Weighted moving average with weight-grid search (w3 > w2 > w1, sum=1).
    Loops only over the ~100-150 valid weight combos, not over rows.
    """
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(clipped_15, window_shape=3, axis=1)
    n = clipped_15.shape[0]

    combos = []
    for w1 in np.round(np.arange(0.15, 0.81, step), 2):
        for w2 in np.round(np.arange(0.25, 0.86 - w1, step), 2):
            w3 = round(1 - (w1 + w2), 2)
            if w3 > w2 > w1:
                combos.append((w1, w2, w3))

    best_rmse = np.full(n, np.inf)
    best_series = np.zeros((n, 13))
    best_weights = np.zeros((n, 3))

    for w1, w2, w3 in combos:
        forecast = (windows[:, :, 0] * w1 + windows[:, :, 1] * w2 + windows[:, :, 2] * w3) / (w1 + w2 + w3)
        rmse = np.abs(d_last - forecast[:, -1])
        better = rmse < best_rmse
        best_rmse = np.where(better, rmse, best_rmse)
        best_series[better] = forecast[better]
        best_weights[better] = [w1, w2, w3]

    return best_series, best_weights

In [4]:
def batch_ewma(clipped_12, alpha=0.4):
    n, T = clipped_12.shape
    ewma = np.zeros((n, T))
    ewma[:, 0] = clipped_12[:, 0]
    for t in range(1, T):
        ewma[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * ewma[:, t - 1]
    forward = ewma[:, -1]
    return np.concatenate([ewma, forward[:, None]], axis=1)

In [5]:
def _batch_poly(clipped_12, degree):
    """Closed-form OLS fit, vectorized across all rows at once.
    The x-grid (1..12) is identical for every row, so a single matrix solve
    replaces thousands of individual sklearn .fit() calls.
    """
    n, T = clipped_12.shape
    x = np.arange(1, T + 1, dtype=float)
    X = np.column_stack([x ** p for p in range(degree + 1)])          # (T, deg+1)
    XtX_inv = np.linalg.pinv(X.T @ X)
    Y = clipped_12.T                                                   # (T, n)
    beta = XtX_inv @ X.T @ Y                                           # (deg+1, n)

    x_full = np.arange(1, T + 2, dtype=float)                          # 13 points
    X_full = np.column_stack([x_full ** p for p in range(degree + 1)])
    pred = X_full @ beta                                                # (13, n)
    return pred.T


def batch_lr(clipped_12):
    return _batch_poly(clipped_12, degree=1)


def batch_pr2(clipped_12):
    return _batch_poly(clipped_12, degree=2)


def batch_pr3(clipped_12):
    return _batch_poly(clipped_12, degree=3)

In [6]:
def batch_ses(clipped_12, alpha=0.8):
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    for t in range(1, T):
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * level[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = clipped_12[:, 0]
    fitted[:, 1:T] = level[:, 0:T - 1]
    fitted[:, T] = level[:, T - 1]
    return fitted


def batch_des(clipped_12, alpha=0.1, beta=0.1):
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    trend = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    trend[:, 0] = clipped_12[:, 1] - clipped_12[:, 0]
    for t in range(1, T):
        prev_pred = level[:, t - 1] + trend[:, t - 1]
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * prev_pred
        trend[:, t] = beta * (level[:, t] - level[:, t - 1]) + (1 - beta) * trend[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = level[:, 0]
    for t in range(1, T):
        fitted[:, t] = level[:, t - 1] + trend[:, t - 1]
    fitted[:, T] = level[:, T - 1] + trend[:, T - 1]
    return fitted

## 3. Demand pattern classification & model eligibility (new)

Picking the "best" model purely by backtest MAE can hand a volatile or level-shifting part a
Poly-2/Poly-3 fit that happens to score well on the backtest window but collapses toward zero
(or an unrealistic curve) on the forward forecast -- mathematically defensible, operationally
wrong.

This section adds, per `(branch, P/N)` row, fully vectorized across all rows at once:

1. **ADI / CV&sup2; classification** (Syntetos-Boylan convention: CV&sup2; computed on
   *non-zero* demand months only) into `Smooth`, `Erratic`, `Intermittent`, `Lumpy`, or `Dead`.
2. **Level-shift detection** -- splits the trailing 12 months into two 6-month halves and flags
   a structural shift when one half's average is 2x (or more) the other's.
3. **Bulk-order outlier detection** -- uses the `C-1..C-12` call-count columns to flag months
   where units-per-call is a robust (median/MAD) outlier vs. that row's own history, i.e. a
   likely one-off bulk PO rather than repeat demand.
4. **Model eligibility matrix** -- restricts the candidate set fed into best-model selection
   based on (1) and (2): trend/curve models (`lr`, `pr2`, `pr3`) are dropped whenever a level
   shift is detected, and further restricted by classification (e.g. `Lumpy` parts are only
   eligible for `ses`, since none of the 8 models are well-suited to lumpy demand -- Croston's
   Method / SBA would be the structurally correct addition if this becomes a large enough
   segment).
5. **Forecast floor / fallback** -- a last-line safety net applied after best-model selection:
   if the chosen model's forward forecast falls below 40% of the trailing 6-month average (and
   that average isn't itself ~0), it's overridden with a recency-weighted fallback instead of
   shipping a near-zero number for a part that's still moving.


In [ ]:
"""
Vectorized demand-pattern classification, level-shift detection, bulk-order
outlier flagging, model-eligibility restriction, and forecast floor/fallback.
All operations run across every row at once (no per-row Python loops), so
this scales to branch-level files with tens of thousands of rows.
"""
import numpy as np

MODEL_NAMES = ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des']

# Model eligibility per SBC classification (before level-shift adjustment)
ELIGIBLE_BY_CLASS = {
    'Smooth':       ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des'],
    'Erratic':      ['wma', 'ewma', 'ses', 'des', 'lr'],
    'Intermittent': ['ma', 'wma', 'ses'],
    'Lumpy':        ['ses'],
    'Dead':         ['ma', 'ses'],
}
TREND_MODELS = ['lr', 'pr2', 'pr3']  # excluded on level shift regardless of class


def classify_demand_pattern(d12, adi_cutoff=1.32, cv2_cutoff=0.49):
    """
    d12: (n, 12) array of the most recent 12 months of demand.
    Returns adi, cv2, classification (all length-n arrays) using the
    standard Syntetos-Boylan convention: CV^2 on non-zero demand sizes only,
    ADI = periods / count of non-zero periods.
    """
    n, T = d12.shape
    count_nz = np.sum(d12 > 0, axis=1)
    count_nz_safe = np.maximum(count_nz, 1)

    adi = T / count_nz_safe
    adi = np.where(count_nz == 0, np.inf, adi)

    sum_nz = np.sum(np.where(d12 > 0, d12, 0), axis=1)
    sumsq_nz = np.sum(np.where(d12 > 0, d12 ** 2, 0), axis=1)
    mean_nz = sum_nz / count_nz_safe
    var_nz = np.maximum(sumsq_nz / count_nz_safe - mean_nz ** 2, 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        cv2 = np.where(mean_nz > 0, var_nz / (mean_nz ** 2), 0.0)
    cv2 = np.where(count_nz < 2, 0.0, cv2)  # can't estimate variance from <2 points

    classification = np.where(
        count_nz == 0, 'Dead',
        np.where(
            (adi < adi_cutoff) & (cv2 < cv2_cutoff), 'Smooth',
            np.where(
                (adi >= adi_cutoff) & (cv2 < cv2_cutoff), 'Intermittent',
                np.where((adi < adi_cutoff) & (cv2 >= cv2_cutoff), 'Erratic', 'Lumpy')
            )
        )
    )
    return adi, cv2, classification


def detect_level_shift(d12, ratio_threshold=2.0):
    """Splits the 12-month window into two 6-month halves and flags a
    structural level shift when one half's average is >= ratio_threshold x
    the other's."""
    first_half = d12[:, :6]
    second_half = d12[:, 6:]
    avg1 = first_half.mean(axis=1)
    avg2 = second_half.mean(axis=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(avg1 > 0, avg2 / np.where(avg1 == 0, np.nan, avg1), np.inf)
    ratio = np.where((avg1 == 0) & (avg2 == 0), 1.0, ratio)

    shifted = (ratio >= ratio_threshold) | ((ratio <= 1 / ratio_threshold) & (avg1 > 0))
    shifted = np.nan_to_num(shifted.astype(float), nan=0.0).astype(bool)
    direction = np.where(avg2 > avg1, 'increasing', np.where(avg2 < avg1, 'decreasing', 'flat'))
    return avg1, avg2, ratio, shifted, direction


def detect_bulk_outliers(d12, c12, z_thresh=1.5):
    """d12/c12: (n,12) demand and calls for the same 12-month window.
    Flags months where units-per-call is a robust (median/MAD) outlier
    vs. that same row's own history -- signals a one-off bulk PO."""
    with np.errstate(divide='ignore', invalid='ignore'):
        upc = np.where(c12 > 0, d12 / np.where(c12 == 0, 1, c12), np.nan)

    median = np.nanmedian(upc, axis=1, keepdims=True)
    mad = np.nanmedian(np.abs(upc - median), axis=1, keepdims=True)
    mad_safe = np.where(mad == 0, 1e-6, mad)
    robust_z = 0.6745 * (upc - median) / mad_safe

    outlier_mask = np.nan_to_num(robust_z, nan=-np.inf) >= z_thresh
    has_outlier = np.any(outlier_mask, axis=1)
    outlier_month_count = np.sum(outlier_mask, axis=1)
    return has_outlier, outlier_month_count


def build_eligibility_matrix(classification, level_shift):
    """Returns (n, 8) boolean matrix: True = model eligible for that row."""
    n = len(classification)
    elig = np.zeros((n, len(MODEL_NAMES)), dtype=bool)
    for cls, models in ELIGIBLE_BY_CLASS.items():
        rows = classification == cls
        for m in models:
            elig[rows, MODEL_NAMES.index(m)] = True
    # level shift: strip trend/curve models regardless of class
    for m in TREND_MODELS:
        idx = MODEL_NAMES.index(m)
        elig[level_shift, idx] = False
    # safety: never leave a row with zero eligible models
    no_elig = ~elig.any(axis=1)
    if no_elig.any():
        elig[no_elig, MODEL_NAMES.index('ses')] = True
    return elig


def select_best_eligible_model(mae_matrix, eligibility):
    """mae_matrix, eligibility: (n, 8). Masks ineligible models to +inf
    before taking the argmin, so the chosen model always comes from the
    eligible set."""
    masked = np.where(eligibility, mae_matrix, np.inf)
    best_idx = np.nanargmin(masked, axis=1)
    return np.array(MODEL_NAMES)[best_idx], best_idx


def apply_forecast_floor(forecast_value, d_last6, floor_pct=0.4):
    """d_last6: (n,6) trailing 6 months of actual demand.
    If forecast falls below floor_pct * trailing 6mo avg (and that avg is
    > 0), override with a recency-weighted (WMA-style) fallback instead of
    shipping a near-zero forecast for a part that's still moving."""
    trailing_avg = d_last6.mean(axis=1)
    floor_value = floor_pct * trailing_avg

    weights = np.arange(1, d_last6.shape[1] + 1, dtype=float)
    wma_fallback = np.average(d_last6, axis=1, weights=weights)

    triggered = (forecast_value < floor_value) & (trailing_avg > 0)
    final = np.where(triggered, wma_fallback, forecast_value)
    return final, triggered, floor_value


if __name__ == "__main__":
    # quick sanity check vs ABC123
    demand = np.array([[2, 0, 2, 1, 0, 5, 7, 10, 9, 6, 2, 7]], dtype=float)
    calls = np.array([[1, 0, 1, 1, 0, 2, 3, 8, 5, 1, 1, 5]], dtype=float)
    adi, cv2, cls = classify_demand_pattern(demand)
    avg1, avg2, ratio, shifted, direction = detect_level_shift(demand)
    has_out, out_count = detect_bulk_outliers(demand, calls)
    elig = build_eligibility_matrix(cls, shifted)
    print("ADI", adi, "CV2", cv2, "class", cls)
    print("shift ratio", ratio, "shifted", shifted, direction)
    print("outlier", has_out, out_count)
    print("eligible models:", np.array(MODEL_NAMES)[elig[0]])


## 4. Metrics, alerts, and the main pipeline

In [7]:
MODEL_NAMES = ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des']


def run_all_models(d_matrix_16, ub, use_fd_window):
    """Fit all 8 models on either the backtest window or the FD window."""
    if use_fd_window:
        actual_12 = np.clip(d_matrix_16[:, -12:], 0, ub[:, None])          # last 12 months
        clipped_15 = np.clip(d_matrix_16[:, -15:], 0, ub[:, None])         # last 15 months
    else:
        actual_12 = np.clip(d_matrix_16[:, -13:-1], 0, ub[:, None])        # 12 months before last
        clipped_15 = np.clip(d_matrix_16[:, :15], 0, ub[:, None])          # first 15 months

    d_last = d_matrix_16[:, -1]

    series = {
        'ma': batch_ma(clipped_15),
        'wma': batch_wma(clipped_15, d_last)[0],
        'ewma': batch_ewma(actual_12),
        'lr': batch_lr(actual_12),
        'pr2': batch_pr2(actual_12),
        'pr3': batch_pr3(actual_12),
        'ses': batch_ses(actual_12),
        'des': batch_des(actual_12),
    }
    return series, actual_12


def compute_metrics(actual_12, series):
    """Vectorized RMSE / MAE / R2 / MAPE / SMAPE per row per model."""
    metrics = {}
    mean_actual = actual_12.mean(axis=1, keepdims=True)
    ss_tot = ((actual_12 - mean_actual) ** 2).sum(axis=1)
    ss_tot_safe = np.where(ss_tot == 0, np.nan, ss_tot)

    for name in MODEL_NAMES:
        pred = series[name][:, :12]
        err = actual_12 - pred
        rmse = np.sqrt((err ** 2).mean(axis=1))
        mae = np.abs(err).mean(axis=1)
        ss_res = (err ** 2).sum(axis=1)
        r2 = 1 - ss_res / ss_tot_safe
        with np.errstate(divide='ignore', invalid='ignore'):
            mape = np.nanmean(np.where(actual_12 == 0, np.nan, np.abs(err) / np.abs(actual_12)), axis=1) * 100
            smape = np.mean(np.abs(err) / (np.abs(actual_12) + np.abs(pred) + 1e-10), axis=1) * 100
        metrics[name] = dict(RMSE=rmse, MAE=mae, R2=r2, MAPE=mape, SMAPE=smape)
    return metrics


def compute_alerts(d_matrix_16):
    last12 = d_matrix_16[:, -12:]
    mean_a = last12.mean(axis=1)
    median_a = np.median(last12, axis=1)
    std_a = last12.std(axis=1)
    max_a = last12.max(axis=1)

    spike = (max_a > np.maximum(mean_a * 1.8, median_a * 1.8)).astype(int)
    with np.errstate(divide='ignore', invalid='ignore'):
        cv = np.where(mean_a != 0, std_a / mean_a, 0)
    volatility = (cv > 1).astype(int)
    zero_count = (last12 == 0).sum(axis=1)
    intermittent = (zero_count >= 6).astype(int)

    score = spike + volatility + intermittent
    label = np.where(score >= 2, 'HIGH RISK', np.where(score == 1, 'REVIEW', 'OK'))
    return spike, volatility, intermittent, score, label

In [ ]:
def find_demand_columns(df):
    cols = [c for c in df.columns if re.match(r'^d-\d+$', c)]
    if not cols:
        raise ValueError(
            "Could not find demand columns matching pattern 'd-<number>' "
            "(e.g. d-1, d-2, ... d-16). Check your column headers."
        )
    return sorted(cols, key=lambda x: int(x.split('-')[1]), reverse=True)


def find_calls_columns(df):
    """Finds C-1..C-12 call-count columns. Optional -- if absent, bulk-order
    outlier detection is skipped and every row's bulk_outlier_flag is False."""
    cols = [c for c in df.columns if re.match(r'^c-\d+$', c)]
    return sorted(cols, key=lambda x: int(x.split('-')[1]), reverse=True)


def find_branch_column(df):
    for candidate in ('brc', 'branch', 'br'):
        if candidate in df.columns:
            return candidate
    raise ValueError("Could not find a branch column (expected 'brc' or 'branch').")


def find_pn_column(df):
    for candidate in ('p/n', 'pn', 'part number'):
        if candidate in df.columns:
            return candidate
    raise ValueError("Could not find a P/N column.")


def find_agc_column(df):
    for candidate in ('agc', 'agency'):
        if candidate in df.columns:
            return candidate
    return None


def forecast(df, include_national_rollup=False,
             adi_cutoff=1.32, cv2_cutoff=0.49,
             level_shift_ratio=2.0, outlier_z=1.5, floor_pct=0.4):
    """Main entry point. Auto-detects branch/P/N/agc/demand/calls columns.

    Output columns match CNH_FD_Processed.ipynb / FD_CNH.ipynb's naming:
    'brc' (not 'branch'). When include_national_rollup=True and an agc
    column is present, two extra tiers of rows are added on top of the raw
    (brc, agc, p/n) rows:
      - brc='National', agc=<real agency>   -> that agency's total across all branches
      - brc='National', agc='ALL AGC'       -> the grand total across every branch
                                                AND every agency, per P/N

    New (Section 3): each row is classified by demand pattern (ADI/CV2),
    checked for a level shift and bulk-order outliers, and best-model
    selection is restricted to the resulting eligible model set. A
    forecast floor/fallback is applied to the final forward forecast.

    adi_cutoff / cv2_cutoff: Syntetos-Boylan classification thresholds.
    level_shift_ratio: half-over-half average ratio that counts as a shift.
    outlier_z: robust z-score threshold for flagging a bulk-order month.
    floor_pct: forecast floor as a fraction of the trailing 6-month average.
    """
    df = df.copy()
    df.columns = [str(c).strip().replace(chr(10), ' ').lower() for c in df.columns]

    branch_col = find_branch_column(df)
    pn_col = find_pn_column(df)
    agc_col = find_agc_column(df)
    demand_cols = find_demand_columns(df)
    calls_cols = find_calls_columns(df)

    df[pn_col] = df[pn_col].astype(str).str.upper().str.strip()

    keep = [branch_col, pn_col] + ([agc_col] if agc_col else []) + demand_cols + calls_cols
    branch_df = df[keep].rename(columns={branch_col: 'brc', pn_col: 'p/n'})
    if agc_col:
        branch_df = branch_df.rename(columns={agc_col: 'agc'})
    else:
        branch_df['agc'] = np.nan

    frames = [branch_df]
    if include_national_rollup:
        if agc_col:
            # Tier 1: national total PER agency -- sums every branch, keeps agc separate
            national_per_agc = branch_df.groupby(['agc', 'p/n'], as_index=False)[demand_cols + calls_cols].sum()
            national_per_agc.insert(0, 'brc', 'National')
            frames.append(national_per_agc[['brc', 'agc', 'p/n'] + demand_cols + calls_cols])

            # Tier 2: grand total across every branch AND every agency for each P/N
            national_all_agc = branch_df.groupby(['p/n'], as_index=False)[demand_cols + calls_cols].sum()
            national_all_agc.insert(0, 'agc', 'ALL AGC')
            national_all_agc.insert(0, 'brc', 'National')
            frames.append(national_all_agc[['brc', 'agc', 'p/n'] + demand_cols + calls_cols])
        else:
            national = branch_df.groupby(['p/n'], as_index=False)[demand_cols + calls_cols].sum()
            national.insert(0, 'brc', 'National')
            frames.append(national[['brc', 'p/n'] + demand_cols + calls_cols])

    out = pd.concat(frames, ignore_index=True)
    d_matrix = out[demand_cols].to_numpy(dtype=float)                                # (n, 16)
    c_matrix = out[calls_cols].to_numpy(dtype=float) if calls_cols else None          # (n, 12)
    n = len(out)

    ub_backtest = d_matrix[:, -13:-1].mean(axis=1) + 1.5 * d_matrix[:, -13:-1].std(axis=1)
    ub_fd = d_matrix[:, -12:].mean(axis=1) + 1.5 * d_matrix[:, -12:].std(axis=1)

    series_bt, actual_bt = run_all_models(d_matrix, ub_backtest, use_fd_window=False)
    metrics_bt = compute_metrics(actual_bt, series_bt)
    mae_matrix = np.column_stack([metrics_bt[m]['MAE'] for m in MODEL_NAMES])

    series_fd, actual_fd = run_all_models(d_matrix, ub_fd, use_fd_window=True)
    metrics_fd = compute_metrics(actual_fd, series_fd)

    # ---- demand pattern classification (Section 3), on the FD 12-month window ----
    d12_fd = d_matrix[:, -12:]
    adi, cv2, classification = classify_demand_pattern(d12_fd, adi_cutoff, cv2_cutoff)
    avg1, avg2, shift_ratio, level_shifted, shift_direction = detect_level_shift(d12_fd, level_shift_ratio)

    if c_matrix is not None:
        has_outlier, outlier_month_count = detect_bulk_outliers(d12_fd, c_matrix, outlier_z)
    else:
        has_outlier = np.zeros(n, dtype=bool)
        outlier_month_count = np.zeros(n, dtype=int)

    eligibility = build_eligibility_matrix(classification, level_shifted)
    best_model, best_idx = select_best_eligible_model(mae_matrix, eligibility)

    result = pd.DataFrame({
        'brc': out['brc'],
        'agc': out['agc'],
        'p/n': out['p/n'],
    })
    result['clipped_d_FD'] = list(actual_fd)
    for name in MODEL_NAMES:
        result[f'{name}_FD'] = list(series_fd[name])

    result['best_model'] = best_model

    fd_forecast_raw = np.array([series_fd[best_model[i]][i, -1] for i in range(n)])

    # ---- forecast floor / fallback safety net ----
    d_last6 = d_matrix[:, -6:]
    fd_forecast_final, floor_triggered, floor_value = apply_forecast_floor(
        fd_forecast_raw, d_last6, floor_pct
    )

    result['FD_forecast_raw'] = fd_forecast_raw
    result['FD_forecast'] = fd_forecast_final
    result['FD_final'] = np.maximum(0, np.round(fd_forecast_final)).astype(int)

    result['metrics_FD'] = [
        [
            {'model': f'{m}_FD', **{k: metrics_fd[m][k][i] for k in ['RMSE', 'MAE', 'R2', 'MAPE', 'SMAPE']}}
            for m in MODEL_NAMES
        ]
        for i in range(n)
    ]
    best_r2_fd = np.array([metrics_fd[best_model[i]]['R2'][i] for i in range(n)])
    result['best_r2_FD'] = best_r2_fd
    result['r2_status_FD'] = np.where(best_r2_fd < 0.25, 'R2 < 0.25', 'Good')

    spike, volatility, intermittent, score, label = compute_alerts(d_matrix)
    result['spike_alert'] = spike
    result['volatility_alert'] = volatility
    result['intermittent_alert'] = intermittent
    result['forecast_alert_score'] = score
    result['forecast_alert_label'] = label

    # ---- classification + model-selection audit trail columns ----
    result['adi'] = np.round(np.where(np.isinf(adi), 999.0, adi), 2)
    result['cv2'] = np.round(cv2, 3)
    result['demand_classification'] = classification
    result['level_shift_detected'] = level_shifted
    result['level_shift_ratio'] = np.round(np.where(np.isinf(shift_ratio), 999.0, shift_ratio), 2)
    result['level_shift_direction'] = shift_direction
    result['bulk_outlier_flag'] = has_outlier
    result['bulk_outlier_months'] = outlier_month_count
    result['eligible_models'] = ['|'.join(np.array(MODEL_NAMES)[eligibility[i]]) for i in range(n)]
    result['excluded_models'] = ['|'.join(np.array(MODEL_NAMES)[~eligibility[i]]) for i in range(n)]
    result['forecast_floor_triggered'] = floor_triggered
    result['forecast_floor_value'] = np.round(floor_value, 2)

    return result


## 5. Run it

Set `INPUT_FILE` to your current `pmovdcE` export each time you run this. `SHEET_NAME` and `SKIP_ROWS` below are set for a multi-agency CNH export (`pmovdcE_CNH_2_Jul_26.xlsx`, sheet "All", header on row 5, i.e. `skiprows=4`) -- change them to match whatever file you're using, or set both to `None` to auto-detect instead.


In [ ]:
INPUT_FILE = "pmovdcE_CNH_19Aug26.xlsx"   # <- change this to your current export
SHEET_NAME = "All"                          # e.g. "All", "CNH Steron", "CNH Non Steron" -- or None to auto-detect
SKIP_ROWS = 4                               # rows before the header row -- or None to auto-detect
INCLUDE_NATIONAL_ROLLUP = False   # set False if you only want branch-level rows

t0 = time.time()
raw = load_movement_data(INPUT_FILE, sheet_name=SHEET_NAME, skiprows=SKIP_ROWS)

t1 = time.time()
result = forecast(raw, include_national_rollup=INCLUDE_NATIONAL_ROLLUP)
print(f"Forecast computed for {len(result)} rows in {time.time()-t1:.1f}s")
if result['agc'].notna().any():
    # sort key as str() since agc can mix real agency codes (int) with the
    # 'ALL AGC' grand-total label (str) once national rollup is included
    agencies = sorted(result['agc'].dropna().unique().tolist(), key=str)
    print(f"Agencies in this file ({len(agencies)}): {agencies}")


In [ ]:
print("Demand classification breakdown:")
print(result['demand_classification'].value_counts())
print()
print("Level shift detected:")
print(result['level_shift_detected'].value_counts())
print()
print("Bulk-order outlier flagged:")
print(result['bulk_outlier_flag'].value_counts())
print()
print("Forecast floor triggered:")
print(result['forecast_floor_triggered'].value_counts())
print()
print("Best model distribution:")
print(result['best_model'].value_counts())


In [10]:
os.makedirs("output", exist_ok=True)
filename = f"output/forecast_branch_{time.strftime('%Y-%m-%d')}.xlsx"
result.to_excel(filename, index=False)
print(f"Saved to {filename}  ({os.path.getsize(filename)/1e6:.1f} MB)")
print(f"Total runtime: {time.time()-t0:.1f}s")

result.head()

Saved to output/forecast_branch_2026-07-10.xlsx  (5.2 MB)
Total runtime: 59.8s


,branch,agc,p/n,clipped_d_FD,ma_FD,wma_FD,ewma_FD,lr_FD,pr2_FD,pr3_FD,...,FD_forecast,FD_final,metrics_FD,best_r2_FD,r2_status_FD,spike_alert,volatility_alert,intermittent_alert,forecast_alert_score,forecast_alert_label
0,20,23,142784 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
1,20,23,15/25B,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
2,20,23,153518 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
3,20,23,153520 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
4,20,23,154272 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
